In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
from calibration import calibration_error
from sklearn.calibration import CalibratedClassifierCV
from joblib import dump



from sklearn.metrics import mean_absolute_percentage_error
def proportion_within_threshold(y_true, y_pred, threshold=0.20):
    return np.mean((np.abs(y_pred - y_true) / y_true) <= threshold)

from sklearn.ensemble import RandomForestRegressor

In [3]:
housing_train = pd.read_parquet(
    "https://lab.cs307.org/housing/data/housing-train.parquet",
)
housing_test = pd.read_parquet(
    "https://lab.cs307.org/housing/data/housing-test.parquet",
)
X_train = housing_train.drop("SalePrice", axis=1)
y_train = housing_train["SalePrice"]

X_test = housing_test.drop("SalePrice", axis=1)
y_test = housing_test["SalePrice"]

In [4]:
X_train.columns.unique()

Index(['MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish', 'Gara

In [ ]:
numeric_transformer = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'),
                                        OneHotEncoder(handle_unknown='ignore', sparse_output=False))

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features)
)

hgb = HistGradientBoostingRegressor()
pipeline = make_pipeline(preprocessor, hgb)

param_grid = {
    'histgradientboostingregressor__learning_rate': [0.03, 0.05, 0.07],
    'histgradientboostingregressor__max_depth': [8, 10, 12],
    'histgradientboostingregressor__max_leaf_nodes': [60, 80, 100],
    'histgradientboostingregressor__min_samples_leaf': [5, 10, 20],
    'histgradientboostingregressor__l2_regularization': [0.001, 0.01, 0.1],
}

model = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    scoring = 'neg_mean_absolute_percentage_error',  
    cv = 3,
    verbose = 2,
    n_jobs = -1
)

model.fit(X_train, y_train)
print(mean_absolute_percentage_error(y_test, model.predict(X_test)), proportion_within_threshold(y_test, model.predict(X_test)))
dump(model, 'housing.joblib')

Fitting 3 folds for each of 243 candidates, totalling 729 fits
0.08954511325445029 0.9253731343283582


['housing.joblib']

In [ ]:
numeric_transformer = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'),
                                        OneHotEncoder(handle_unknown='ignore', sparse_output=False))

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features)
)

hgb = HistGradientBoostingRegressor()
pipeline = make_pipeline(preprocessor, hgb)

param_grid = {
    'histgradientboostingregressor__learning_rate': [0.03, 0.05, 0.07],
    'histgradientboostingregressor__max_depth': [8, 10, 12],
    'histgradientboostingregressor__max_leaf_nodes': [60, 80, 100],
    'histgradientboostingregressor__min_samples_leaf': [5, 10, 20],
    'histgradientboostingregressor__l2_regularization': [0.001, 0.01, 0.1],
    'histgradientboostingregressor__loss': ['absolute_error']
}

model2 = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    scoring='neg_mean_absolute_percentage_error',  
    cv=3,
    verbose=2,
    n_jobs=-1
)

model2.fit(X_train, y_train)
print(mean_absolute_percentage_error(y_test, model2.predict(X_test)), proportion_within_threshold(y_test, model2.predict(X_test)))

Fitting 3 folds for each of 243 candidates, totalling 729 fits
0.08926424753327264 0.9147121535181236


In [11]:
X_train2 = X_train.copy() 
X_test2 = X_test.copy()
X_train2 = X_train2.drop(columns=["Order", "PID"])
X_test2 = X_test2.drop(columns=["Order", "PID"])

numeric_transformer = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'),
                                        OneHotEncoder(handle_unknown='ignore', sparse_output=False))

numeric_features = X_train2.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train2.select_dtypes(include=['object', 'category']).columns

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features)
)

hgb = HistGradientBoostingRegressor()
pipeline = make_pipeline(preprocessor, hgb)

param_grid = {
    'histgradientboostingregressor__learning_rate': [0.03, 0.04, 0.05],
    'histgradientboostingregressor__max_depth': [6, 8, 10],
    'histgradientboostingregressor__max_leaf_nodes': [55, 65, 75],
    'histgradientboostingregressor__min_samples_leaf': [8, 10, 12],
    'histgradientboostingregressor__l2_regularization': [0.003, 0.01, 0.03],
    'histgradientboostingregressor__loss': ['absolute_error']
}


model3 = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    scoring='neg_mean_absolute_percentage_error',  
    cv=3,
    verbose=2,
    n_jobs=-1
)

model3.fit(X_train2, y_train)
print(mean_absolute_percentage_error(y_test, model3.predict(X_test2)), proportion_within_threshold(y_test, model3.predict(X_test2)))

Fitting 3 folds for each of 243 candidates, totalling 729 fits
0.09436570725002083 0.906183368869936


In [5]:
numeric_transformer = make_pipeline(SimpleImputer(strategy='mean'), StandardScaler())
categorical_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'),
                                        OneHotEncoder(handle_unknown='ignore', sparse_output=False))

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features)
)

hgb = HistGradientBoostingRegressor()
pipeline = make_pipeline(preprocessor, hgb)

param_grid = {
    'histgradientboostingregressor__learning_rate': [0.03, 0.05, 0.07],
    'histgradientboostingregressor__max_depth': [8, 10, 12],
    'histgradientboostingregressor__max_leaf_nodes': [60, 80, 100],
    'histgradientboostingregressor__min_samples_leaf': [5, 10, 20],
    'histgradientboostingregressor__l2_regularization': [0.001, 0.01, 0.1],
}

gs = GridSearchCV(
    estimator = pipeline,
    param_grid = param_grid,
    scoring = 'neg_mean_absolute_percentage_error',
    cv = 3,
    n_jobs = -1,
    verbose = 2
)
gs.fit(X_train, y_train)
print(gs.best_params_)


best_hgb = HistGradientBoostingRegressor(
    learning_rate = gs.best_params_['histgradientboostingregressor__learning_rate'],
    max_depth = gs.best_params_['histgradientboostingregressor__max_depth'],
    max_leaf_nodes = gs.best_params_['histgradientboostingregressor__max_leaf_nodes'],
    min_samples_leaf = gs.best_params_['histgradientboostingregressor__min_samples_leaf'],
    l2_regularization = gs.best_params_['histgradientboostingregressor__l2_regularization'],
    max_iter = 800,                     # NOT tuned — boosts performance
    early_stopping = True,
    validation_fraction = 0.1,
    random_state = 77
)

final_pipeline = make_pipeline(preprocessor, best_hgb)
final_pipeline.fit(X_train, y_train)

Fitting 3 folds for each of 243 candidates, totalling 729 fits
{'histgradientboostingregressor__l2_regularization': 0.01, 'histgradientboostingregressor__learning_rate': 0.07, 'histgradientboostingregressor__max_depth': 12, 'histgradientboostingregressor__max_leaf_nodes': 60, 'histgradientboostingregressor__min_samples_leaf': 20}


,steps,"[('columntransformer', ...), ('histgradientboostingregressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [6]:
print(mean_absolute_percentage_error(y_test, final_pipeline.predict(X_test)), proportion_within_threshold(y_test, final_pipeline.predict(X_test)))

0.08964758084165697 0.9232409381663113
